In [109]:
import feedparser
import pandas as pd
import time
url = ["https://www.cert.ssi.gouv.fr/avis/feed/", "https://www.cert.ssi.gouv.fr/alerte/feed/"]
rss_feeds = [feedparser.parse(u) for u in url]

In [ ]:
import requests
import re
entres = {}

for rss_feed in rss_feeds:
    for entry in rss_feed.entries:
        link = entry.link
        response = requests.get(link + "/json/")

        ref_anssi = entry.link.split("/")[:-1][-1]

        if response.status_code == 200:
            data = response.json()
            if len(data.get('cves')) > 0:
                cves = [cve['name'] for cve in data.get('cves', [])]
                print(f"CVEs for {ref_anssi}: {cves}")
                entres[ref_anssi] = cves
        # time.sleep(1)  # Pause de 1 seconde entre les requêtes



    

CVEs for CERTFR-2026-AVI-0632: ['CVE-2026-6517']
CVEs for CERTFR-2026-AVI-0716: ['CVE-2026-45257', 'CVE-2026-49413']
CVEs for CERTFR-2026-AVI-0742: ['CVE-2025-47907', 'CVE-2024-12905', 'CVE-2025-22870', 'CVE-2023-24531']
CVEs for CERTFR-2026-AVI-0743: ['CVE-2026-9266']
CVEs for CERTFR-2026-AVI-0744: ['CVE-2026-47825', 'CVE-2026-41708']
CVEs for CERTFR-2026-AVI-0745: ['CVE-2026-31483', 'CVE-2026-43414', 'CVE-2026-31493', 'CVE-2026-31402', 'CVE-2026-45852', 'CVE-2026-31758', 'CVE-2026-31685', 'CVE-2026-45910', 'CVE-2026-31405', 'CVE-2026-43054', 'CVE-2023-20585', 'CVE-2026-31473', 'CVE-2026-31613', 'CVE-2026-46114', 'CVE-2026-23380', 'CVE-2026-43284', 'CVE-2026-43362', 'CVE-2026-23271', 'CVE-2026-31614', 'CVE-2026-46113', 'CVE-2026-3150', 'CVE-2026-31568', 'CVE-2026-31516', 'CVE-2026-23317', 'CVE-2026-43012', 'CVE-2026-43503', 'CVE-2026-43009', 'CVE-2026-43499', 'CVE-2026-23359', 'CVE-2026-46043', 'CVE-2026-43252', 'CVE-2026-23437', 'CVE-2026-46243', 'CVE-2026-43360', 'CVE-2026-43328', '

In [111]:
cvss = {ref_anssi: {} for ref_anssi in entres.keys()}
cwe = {ref_anssi: {} for ref_anssi in entres.keys()}
for ref_anssi, cves in entres.items():
    print(f"Ref ANSSI: {ref_anssi}, CVEs: {cves}")
    for cve in cves:
        cve_url = f"https://cveawg.mitre.org/api/cve/{cve}"
        cve_response = requests.get(cve_url)
        if cve_response.status_code == 200:
            cve_data = cve_response.json()
            cwe_data = cve_data.get("containers", {}).get("cna", {}).get("problemTypes", [{}])[0].get("descriptions", [{}])[0].get("cweId")

            cvss[ref_anssi][cve] = None
            cwe[ref_anssi][cve] = cwe_data

            try:
                metrics_list = cve_data.get("containers", {}).get("cna", {}).get("metrics", [])

                if metrics_list:
                    cvss_data = metrics_list[0].get("cvssV4_0", {})
                    
                    base_score = cvss_data.get("baseScore")
                    cvss[ref_anssi][cve] = base_score
                    cwe[ref_anssi][cve] = cwe_data
                    
                    print(f"  -> CVE: {cve} | Base Score CVSS 4.0: {base_score} | CWE: {cwe_data}")
                else:
                    print(f"  -> CVE: {cve} | Aucune métrique trouvée.")
                    
            except KeyError as e:
                print(f"  -> Erreur lors de la lecture des métriques pour {cve}: {e}")
        # time.sleep(1)  # Pause de 1 seconde entre les requêtes
            

Ref ANSSI: CERTFR-2026-AVI-0632, CVEs: ['CVE-2026-6517']
  -> CVE: CVE-2026-6517 | Base Score CVSS 4.0: None | CWE: CWE-522
Ref ANSSI: CERTFR-2026-AVI-0716, CVEs: ['CVE-2026-45257', 'CVE-2026-49413']
Ref ANSSI: CERTFR-2026-AVI-0742, CVEs: ['CVE-2025-47907', 'CVE-2024-12905', 'CVE-2025-22870', 'CVE-2023-24531']
  -> CVE: CVE-2025-47907 | Aucune métrique trouvée.
  -> CVE: CVE-2024-12905 | Base Score CVSS 4.0: None | CWE: CWE-59
  -> CVE: CVE-2025-22870 | Aucune métrique trouvée.
  -> CVE: CVE-2023-24531 | Aucune métrique trouvée.
Ref ANSSI: CERTFR-2026-AVI-0743, CVEs: ['CVE-2026-9266']
  -> CVE: CVE-2026-9266 | Base Score CVSS 4.0: 7 | CWE: CWE-325
Ref ANSSI: CERTFR-2026-AVI-0744, CVEs: ['CVE-2026-47825', 'CVE-2026-41708']
  -> CVE: CVE-2026-47825 | Base Score CVSS 4.0: None | CWE: CWE-346
  -> CVE: CVE-2026-41708 | Base Score CVSS 4.0: None | CWE: CWE-400
Ref ANSSI: CERTFR-2026-AVI-0745, CVEs: ['CVE-2026-31483', 'CVE-2026-43414', 'CVE-2026-31493', 'CVE-2026-31402', 'CVE-2026-45852', 'C

In [112]:
epss = {ref_anssi: {} for ref_anssi in entres.keys()}
for ref_anssi, cves in entres.items():
    print(f"Ref ANSSI: {ref_anssi}, CVEs: {cves}")
    for cve in cves:
        cve_url = f"https://api.first.org/data/v1/epss?cve={cve}"
        cve_response = requests.get(cve_url)
        epss[ref_anssi][cve] = None  # Initialiser à None par défaut
        if cve_response.status_code == 200:
            cve_data = cve_response.json()
            # print(f"  -> CVE: {cve} | EPS Score Data: {cve_data}")
            try:
                epss_score = cve_data.get("data", [{}])[0].get("epss")
                epss[ref_anssi][cve] = epss_score
                print(f"  -> CVE: {cve} | EPS Score: {epss_score}")
            except IndexError as e:
                print(f"  -> Erreur lors de la lecture du score EPS pour {cve}: {e}")
        # time.sleep(1)  # Pause de 1 seconde entre les requêtes

Ref ANSSI: CERTFR-2026-AVI-0632, CVEs: ['CVE-2026-6517']
  -> CVE: CVE-2026-6517 | EPS Score: 0.001860000
Ref ANSSI: CERTFR-2026-AVI-0716, CVEs: ['CVE-2026-45257', 'CVE-2026-49413']
  -> Erreur lors de la lecture du score EPS pour CVE-2026-45257: list index out of range
  -> Erreur lors de la lecture du score EPS pour CVE-2026-49413: list index out of range
Ref ANSSI: CERTFR-2026-AVI-0742, CVEs: ['CVE-2025-47907', 'CVE-2024-12905', 'CVE-2025-22870', 'CVE-2023-24531']
  -> CVE: CVE-2025-47907 | EPS Score: 0.003310000
  -> CVE: CVE-2024-12905 | EPS Score: 0.018950000
  -> CVE: CVE-2025-22870 | EPS Score: 0.003500000
  -> CVE: CVE-2023-24531 | EPS Score: 0.008330000
Ref ANSSI: CERTFR-2026-AVI-0743, CVEs: ['CVE-2026-9266']
  -> CVE: CVE-2026-9266 | EPS Score: 0.000700000
Ref ANSSI: CERTFR-2026-AVI-0744, CVEs: ['CVE-2026-47825', 'CVE-2026-41708']
  -> CVE: CVE-2026-47825 | EPS Score: 0.001860000
  -> CVE: CVE-2026-41708 | EPS Score: 0.004600000
Ref ANSSI: CERTFR-2026-AVI-0745, CVEs: ['CVE-2

In [113]:
data_list = []

for ref_anssi, cves in entres.items():
    for cve in cves:
        data_list.append({
            'REF_ANSSI': ref_anssi,
            'CVE': cve,
            'CVSS_Score': cvss.get(ref_anssi, {}).get(cve) if cvss.get(ref_anssi, {}).get(cve) is not None else None,
            'CWE': cwe.get(ref_anssi, {}).get(cve) if cwe.get(ref_anssi, {}).get(cve) is not None else None,
            'EPSS_Score': epss.get(ref_anssi, {}).get(cve) if epss.get(ref_anssi, {}).get(cve) is not None else None
        })

# Création du DataFrame
df_vulnerabilities = pd.DataFrame(data_list)

# Enregistrement en CSV
df_vulnerabilities.to_csv('data.csv', index=False, encoding='utf-8', sep=';')

print("Données enregistrées dans 'data.csv'")
df_vulnerabilities.head()

Données enregistrées dans 'data.csv'


,REF_ANSSI,CVE,CVSS_Score,CWE,EPSS_Score
0,CERTFR-2026-AVI-0632,CVE-2026-6517,NaN,CWE-522,0.001860000
1,CERTFR-2026-AVI-0716,CVE-2026-45257,NaN,NaN,NaN
2,CERTFR-2026-AVI-0716,CVE-2026-49413,NaN,NaN,NaN
3,CERTFR-2026-AVI-0742,CVE-2025-47907,NaN,NaN,0.003310000
4,CERTFR-2026-AVI-0742,CVE-2024-12905,NaN,CWE-59,0.018950000


In [114]:
df_vulnerabilities.isna().sum()

REF_ANSSI       0
CVE             0
CVSS_Score    678
CWE           467
EPSS_Score     11
dtype: int64